In [1]:
from pathlib import Path
import xarray as xr

PROJECT_ROOT = Path.cwd().parent
ds = xr.open_dataset(PROJECT_ROOT / "data" / "raw" / "weather" / "era5_nord_torino.nc")
ds

<xarray.Dataset> Size: 9MB
Dimensions:     (valid_time: 289272)
Coordinates:
  * valid_time  (valid_time) datetime64[ns] 2MB 1991-01-01 ... 2023-12-31T23:...
    latitude    float64 8B ...
    longitude   float64 8B ...
Data variables:
    u100        (valid_time) float32 1MB ...
    v100        (valid_time) float32 1MB ...
    t2m         (valid_time) float32 1MB ...
    sp          (valid_time) float32 1MB ...
    ssrd        (valid_time) float32 1MB ...
    fdir        (valid_time) float32 1MB ...
Attributes:
    Conventions:             CF-1.7
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_edition:            1
    GRIB_subCentre:          0
    history:                 2024-09-02T04:48 GRIB to CDM+CF via cfgrib-0.9.1...
    institution:             European Centre for Medium-Range Weather Forecasts

In [2]:
df = ds.to_dataframe().reset_index().set_index("valid_time")
df.describe()

,u100,v100,t2m,sp,ssrd,fdir,latitude,longitude
count,289272.000000,289272.000000,289272.000000,289272.000000,2.892720e+05,2.892720e+05,289272.0,289272.00
mean,-0.247541,-0.624430,286.386322,97709.945312,5.542896e+05,3.625907e+05,45.0,7.75
std,1.566761,1.368941,8.591470,735.689026,8.189976e+05,6.200802e+05,0.0,0.00
min,-10.799561,-10.565903,254.523712,94021.531250,-1.901566e+00,0.000000e+00,45.0,7.75
25%,-1.286377,-1.402206,279.480209,97291.341797,0.000000e+00,0.000000e+00,45.0,7.75
50%,-0.450775,-0.540199,286.236115,97732.390625,2.374400e+04,6.400000e+01,45.0,7.75
75%,0.576309,0.227066,293.029053,98161.371094,9.411840e+05,4.972800e+05,45.0,7.75
max,9.828796,7.164673,312.098328,100419.812500,3.497152e+06,3.055488e+06,45.0,7.75


In [3]:
import numpy as np
import pandas as pd
conv = pd.DataFrame(index=df.index)
conv["temp_c"] = df["t2m"] - 273.15
conv["ghi_wm2"] = df["ssrd"] / 3600
conv["bhi_wm2"] = df["fdir"] / 3600
conv["pressure_pa"] = df["sp"]
conv["wind_speed_ms"] = np.sqrt(df["u100"]**2 + df["v100"]**2)

conv.describe()

conv["ghi_wm2"].nsmallest(10)

valid_time
2016-10-10 00:00:00   -5.282126e-04
2011-12-16 21:00:00   -3.237526e-04
2004-12-04 05:00:00   -3.972126e-07
1991-01-01 00:00:00    0.000000e+00
1991-01-01 01:00:00    0.000000e+00
1991-01-01 02:00:00    0.000000e+00
1991-01-01 03:00:00    0.000000e+00
1991-01-01 04:00:00    0.000000e+00
1991-01-01 05:00:00    0.000000e+00
1991-01-01 06:00:00    0.000000e+00
Name: ghi_wm2, dtype: float32

In [4]:
conv = pd.DataFrame(index=df.index)
conv["temp_c"] = df["t2m"] - 273.15
conv["ghi_wm2"] = (df["ssrd"] / 3600).clip(lower=0)
conv["bhi_wm2"] = (df["fdir"] / 3600).clip(lower=0)
conv["pressure_pa"] = df["sp"]
conv["wind_speed_ms"] = np.sqrt(df["u100"]**2 + df["v100"]**2)

conv_15min = conv.resample("15min").asfreq().interpolate(method="time")
conv_15min.index = conv_15min.index.tz_localize("UTC").tz_convert("Europe/Rome")

conv_15min

len(conv_15min)
conv_15min.isna().sum()

temp_c           0
ghi_wm2          0
bhi_wm2          0
pressure_pa      0
wind_speed_ms    0
dtype: int64

In [5]:
def process_city(path):
    ds = xr.open_dataset(path)
    df = ds.to_dataframe().reset_index().set_index("valid_time")

    conv = pd.DataFrame(index=df.index)
    conv["temp_c"] = df["t2m"] - 273.15
    conv["ghi_wm2"] = (df["ssrd"] / 3600).clip(lower=0)
    conv["bhi_wm2"] = (df["fdir"] / 3600).clip(lower=0)
    conv["pressure_pa"] = df["sp"]
    conv["wind_speed_ms"] = np.sqrt(df["u100"]**2 + df["v100"]**2)

    conv_15min = conv.resample("15min").asfreq().interpolate(method="time")
    conv_15min.index = conv_15min.index.tz_localize("UTC").tz_convert("Europe/Rome")
    return conv_15min

In [6]:
nord_cities = ["torino", "milano", "verona", "bologna"]
city_data = {
    city: process_city(PROJECT_ROOT / "data" / "raw" / "weather" / f"era5_nord_{city}.nc")
    for city in nord_cities
}

In [7]:
stacked = pd.concat(city_data.values(), keys=city_data.keys())
zone_weather = stacked.groupby(level=1).mean()
zone_spread = stacked.groupby(level=1).std(ddof=0)

zone_weather

,temp_c,ghi_wm2,bhi_wm2,pressure_pa,wind_speed_ms
valid_time,,,,,
1991-01-01 01:00:00+01:00,4.696747,0.0,0.0,99244.109375,0.866254
1991-01-01 01:15:00+01:00,4.717133,0.0,0.0,99224.578125,0.893586
1991-01-01 01:30:00+01:00,4.737518,0.0,0.0,99205.054688,0.920918
1991-01-01 01:45:00+01:00,4.757904,0.0,0.0,99185.531250,0.948250
1991-01-01 02:00:00+01:00,4.778290,0.0,0.0,99166.000000,0.975581
...,...,...,...,...,...
2023-12-31 23:00:00+01:00,7.023560,0.0,0.0,98278.101562,3.135315
2023-12-31 23:15:00+01:00,6.938110,0.0,0.0,98286.328125,3.118121
2023-12-31 23:30:00+01:00,6.852661,0.0,0.0,98294.562500,3.100926


In [8]:
zone_spread.loc["2020-07-15 13:00"]

temp_c              0.442971
ghi_wm2            97.499886
bhi_wm2           108.725136
pressure_pa      1121.626221
wind_speed_ms       0.831868
Name: 2020-07-15 13:00:00+02:00, dtype: float32

In [9]:
import yaml
with open(PROJECT_ROOT / "config.yaml") as f:
    config = yaml.safe_load(f)

sud_cities = [city["name"] for city in config["zones"]["sud"]["cities"]]

In [10]:
sud_city_data = {
    city: process_city(PROJECT_ROOT / "data" / "raw" / "weather" / f"era5_sud_{city}.nc")
    for city in sud_cities
}

In [11]:
sud_stacked = pd.concat(sud_city_data.values(), keys=sud_city_data.keys())
sud_zone_weather = sud_stacked.groupby(level=1).mean()
sud_zone_spread = sud_stacked.groupby(level=1).std(ddof=0)

sud_zone_weather

,temp_c,ghi_wm2,bhi_wm2,pressure_pa,wind_speed_ms
valid_time,,,,,
1991-01-01 01:00:00+01:00,6.483856,0.0,0.0,97388.609375,3.214530
1991-01-01 01:15:00+01:00,6.495575,0.0,0.0,97384.890625,3.317025
1991-01-01 01:30:00+01:00,6.507294,0.0,0.0,97381.179688,3.419520
1991-01-01 01:45:00+01:00,6.519012,0.0,0.0,97377.468750,3.522015
1991-01-01 02:00:00+01:00,6.530731,0.0,0.0,97373.750000,3.624510
...,...,...,...,...,...
2023-12-31 23:00:00+01:00,10.155396,0.0,0.0,96989.601562,6.009001
2023-12-31 23:15:00+01:00,10.136108,0.0,0.0,96980.890625,5.988015
2023-12-31 23:30:00+01:00,10.116821,0.0,0.0,96972.187500,5.967030


In [12]:
len(sud_zone_weather)
sud_zone_weather.isna().sum()
sud_zone_weather.loc["2020-07-15 13:00"]
sud_zone_spread.loc["2020-07-15 13:00"]

temp_c              2.028373
ghi_wm2            91.069420
bhi_wm2           114.511246
pressure_pa      3225.948242
wind_speed_ms       1.140368
Name: 2020-07-15 13:00:00+02:00, dtype: float32

In [13]:
def validate_zone_weather(df, zone_name, start_year, end_year):
    if df.isna().any().any():
        raise ValueError(f"{zone_name}: NaN values found in processed weather data")

    if not df["temp_c"].between(-50, 60).all():
        raise ValueError(f"{zone_name}: temperature outside plausible range (-50 to 60 C)")

    if not df["pressure_pa"].between(70_000, 110_000).all():
        raise ValueError(f"{zone_name}: pressure outside plausible range (70,000-110,000 Pa)")

    num_years = end_year - start_year + 1

In [14]:
def process_city(path):
    ds = xr.open_dataset(path)
    df = ds.to_dataframe().reset_index()

    time_col = "valid_time" if "valid_time" in df.columns else "time"
    df = df.set_index(time_col)

    conv = pd.DataFrame(index=df.index)
    conv["temp_c"] = df["t2m"] - 273.15
    conv["ghi_wm2"] = (df["ssrd"] / 3600).clip(lower=0)
    conv["bhi_wm2"] = (df["fdir"] / 3600).clip(lower=0)
    conv["pressure_pa"] = df["sp"]
    conv["wind_speed_ms"] = np.sqrt(df["u100"]**2 + df["v100"]**2)

    conv_15min = conv.resample("15min").asfreq().interpolate(method="time")
    conv_15min.index = conv_15min.index.tz_localize("UTC").tz_convert("Europe/Rome")
    return conv_15min


In [15]:
def aggregate_zone(city_data, zone_name):
    common_index = city_data[next(iter(city_data))].index
    for df in city_data.values():
        common_index = common_index.intersection(df.index)

    if len(common_index) == 0:
        raise ValueError(f"{zone_name}: no overlapping timestamps across cities, check for corrupted/misaligned downloads")

    stacked = pd.concat(city_data.values(), keys=city_data.keys())
    zone_weather = stacked.groupby(level=1).mean()
    zone_spread = stacked.groupby(level=1).std(ddof=0)

    validate_zone_weather(zone_weather, zone_name)
    return zone_weather, zone_spread

In [16]:
def zone_summary_row(zone_name, zone_weather, zone_spread):
    snapshot = zone_weather.loc["2020-07-15 13:00"]
    spread_snapshot = zone_spread.loc["2020-07-15 13:00"]
    return {
        "zone": zone_name,
        "annual_mean_ghi_wm2": zone_weather["ghi_wm2"].mean(),
        "annual_mean_temp_c": zone_weather["temp_c"].mean(),
        "summer_midday_ghi_wm2": snapshot["ghi_wm2"],
        "summer_midday_temp_c": snapshot["temp_c"],
        "pressure_spread_pa": spread_snapshot["pressure_pa"],
    }

In [17]:
validate_zone_weather(zone_weather, "nord", config["weather"]["start_year"], config["weather"]["end_year"])
validate_zone_weather(sud_zone_weather, "sud", config["weather"]["start_year"], config["weather"]["end_year"])


In [18]:
summary_rows = [
    zone_summary_row("nord", zone_weather, zone_spread),
    zone_summary_row("sud", sud_zone_weather, sud_zone_spread),
]
pd.DataFrame(summary_rows)

,zone,annual_mean_ghi_wm2,annual_mean_temp_c,summer_midday_ghi_wm2,summer_midday_temp_c,pressure_spread_pa
0,nord,159.000961,13.825830,623.942261,25.206940,1121.626221
1,sud,183.863220,15.105055,845.924438,26.664948,3225.948242
